# **GroupDNA - WhatsApp Chat Analyser**

> ### **Minor Project**

> ### **By Somyajeet Satapathy**

> ### **Dataset : hostel_bois.txt**




**------------------------------------------------------------------------------------------------------**

###**Loading the WhatsApp Chat Dataset**

In [220]:
import numpy as np     # for heatmap calculations
from datetime import datetime

In [221]:
from google.colab import drive
drive.mount('/content/drive')         # Mount Google Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [222]:
file_path = "/content/drive/MyDrive/Unlox/Projects/Minor Project-1/hostel_bois.txt"

# Read all chat lines from the exported file
with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Dataset loaded successfully!")
print(f"Total chat lines: {len(lines)}")   # Verify dataset size

Dataset loaded successfully!
Total chat lines: 3178


✔ **Uploading the Dataset Completed Successfully**

### **Feature 1: WhatsApp Chat Parser**

This feature is used to read the exported WhatsApp chat file and convert the raw chat data into a structured format for further analysis.

In [223]:
# Containers for storing parsed chat data

chat_data = []                 # Stores every parsed message
participants = set()           # Stores unique participant names

system_messages = 0            # Counting system notifications
media_messages = 0             # Counting media placeholders
deleted_messages = 0           # Counting deleted messages

**Separate Timestamp and Chat Content**

In [224]:
# Traversing through every line in the chat
for line in lines:
    line = line.strip()                          # Leading and trailing spaces are removed
    if not line:
        continue                                # Empty lines are skipped
    if " - " not in line:
        continue                                # Invalid chat lines are ignored

    # Separating timestamp and chat content
    date_time, chat_text = line.split(" - ", 1)

    try:
        timestamp = datetime.strptime(date_time, "%d/%m/%y, %H:%M")
    except ValueError:
        continue                                # Invalid timestamp formats are ignored

    # Identifying system messages
    if ": " not in chat_text:
        sender = "System"
        message = chat_text
        system_messages += 1
    else:
        # Extracting sender and message details
        sender, message = chat_text.split(": ", 1)
        sender = sender.strip()                 # Extra spaces are removed
        message = message.strip()
        participants.add(sender)                # Unique participants are stored

    # Detecting media messages
    if message == "<Media omitted>":
        media_messages += 1
        continue                                     # Media records are excluded

    # Detecting deleted messages
    if message == "This message was deleted":
        deleted_messages += 1
        continue                                     # Deleted records are excluded

    # Skipping system messages
    if sender == "System":
        continue                                     # System records are excluded

    # Storing valid chat records
    chat_data.append({
        "timestamp": timestamp,
        "sender": sender,
        "message": message
    })

**Verifying Parsed Chat Data**

In [225]:
print("=" * 55)
print("PARSER SUMMARY")
print("=" * 55)

print(f"Total Chat Lines      : {len(lines)}")
print(f"Valid Messages        : {len(chat_data)}")
print(f"Participants          : {len(participants)}")
print(f"System Messages       : {system_messages}")
print(f"Media Messages        : {media_messages}")
print(f"Deleted Messages      : {deleted_messages}")

print("\nFirst 5 Valid Messagess")
print("-" * 55)

for record in chat_data[:5]:
    print(record)

PARSER SUMMARY
Total Chat Lines      : 3178
Valid Messages        : 3127
Participants          : 6
System Messages       : 4
Media Messages        : 32
Deleted Messages      : 15

First 5 Valid Messagess
-------------------------------------------------------
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'message': 'scene fix'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'message': 'haan'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 18), 'sender': 'Rahul', 'message': 'kya scene'}
{'timestamp': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'message': 'abhi free hai?'}
{'timestamp': datetime.datetime(2024, 4, 1, 2, 13), 'sender': 'Rahul', 'message': 'abey'}


✔ **Feature 1 Completed Successfully**

### **Feature 2: Group Overview**

This feature provides a summary of the WhatsApp group by calculating the
total number of messages, participants, chat duration, and individual
message contributions.

**Calculating Group Statistics**

In [226]:
# Calculating overall group statistics
total_messages = len(chat_data)
total_participants = len(participants)

start_date = min(record["timestamp"] for record in chat_data)      # Earliest message date is obtained
end_date = max(record["timestamp"] for record in chat_data)        # Latest message date is obtained

message_count = {}                                                 # Participant-wise message count is initialized

**Counting Participant Messages**

In [227]:
# Counting messages sent by each participant
for record in chat_data:
    if record["sender"] == "System":
        continue                                 # System messages are excluded
    sender = record["sender"]

    if sender not in message_count:
        message_count[sender] = 0

    message_count[sender] += 1                   # Message count is updated

**Displaying Group Overview**

In [228]:
print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)

# Calculating total user messages
total_user_messages = sum(message_count.values())      # Total valid messages are calculated

print(f"Total Valid Messages : {len(chat_data)}")
print(f"Participants        : {total_participants}")
print(f"Chat Started        : {start_date.strftime('%d %b %Y')}")
print(f"Latest Message      : {end_date.strftime('%d %b %Y')}")

print("\nParticipant Contributions")
print("-" * 60)

for sender, count in sorted(message_count.items(),
                            key=lambda item: item[1],
                            reverse=True):

    percentage = (count / total_user_messages) * 100      # Contribution percentage is calculated

    print(f"{sender:<15} {count:>5} messages ({percentage:.2f}%)")

GROUP OVERVIEW
Total Valid Messages : 3127
Participants        : 6
Chat Started        : 01 Apr 2024
Latest Message      : 30 May 2024

Participant Contributions
------------------------------------------------------------
Rahul             940 messages (30.06%)
Priya             712 messages (22.77%)
Neha              624 messages (19.96%)
Aman              484 messages (15.48%)
Karan             345 messages (11.03%)
Vikas              22 messages (0.70%)


**Calculating Participant Statistics**

In [229]:
# Initializing participant statistics
word_count = {}
average_length = {}

# Calculating statistics for every participant
for sender in participants:

    total_words = 0
    total_messages = 0

    for record in chat_data:

        if record["sender"] != sender:
            continue                              # Other participants are ignored

        total_messages += 1

        words = record["message"].split()

        total_words += len(words)

    word_count[sender] = total_words             # Word count is stored

    if total_messages > 0:
        average_length[sender] = total_words / total_messages
    else:
        average_length[sender] = 0

**Displaying Participant Statistics**

In [230]:
print("=" * 75)
print("PARTICIPANT STATISTICS")
print("=" * 75)

print(f"{'Member':<15}{'Words':>10}{'Avg Length':>18}")
print("-" * 75)

for sender in sorted(participants):

    print(f"{sender:<15}{word_count[sender]:>10}{average_length[sender]:>18.2f}")

PARTICIPANT STATISTICS
Member              Words        Avg Length
---------------------------------------------------------------------------
Aman                 2430              5.02
Karan               19681             57.05
Neha                 3317              5.32
Priya                3560              5.00
Rahul                2399              2.55
Vikas                  40              1.82


**Verifying Group Overview**

In [231]:
print("=" * 60)
print("FEATURE 2 VERIFICATION")
print("=" * 60)

most_active = max(message_count, key=message_count.get)

print(f"Most Active Member : {most_active}")
print(f"Messages Sent      : {message_count[most_active]}")
print(f"Total Words        : {word_count[most_active]}")

FEATURE 2 VERIFICATION
Most Active Member : Rahul
Messages Sent      : 940
Total Words        : 2399


✔ **Feature 2 Completed Successfully**

###**Feature 3: Activity Analysis**

This feature analyzes the activity pattern of the WhatsApp group by
identifying the busiest day and busiest hour based on the number of
messages exchanged.

**Counting Daily and Hourly Activity**

In [232]:
# Initializing activity counters
daily_activity = {}
hourly_activity = {}

# Counting messages by date and hour
for record in chat_data:

    date = record["timestamp"].date()
    hour = record["timestamp"].hour

    if date not in daily_activity:
        daily_activity[date] = 0

    if hour not in hourly_activity:
        hourly_activity[hour] = 0

    daily_activity[date] += 1                    # Daily message count is updated
    hourly_activity[hour] += 1                   # Hourly message count is updated

**Identifying Peak Activity**

In [233]:
# Identifying the busiest day and hour
busiest_day = max(daily_activity, key=daily_activity.get)
busiest_hour = max(hourly_activity, key=hourly_activity.get)

day_messages = daily_activity[busiest_day]
hour_messages = hourly_activity[busiest_hour]

**Displaying Activity Analysis**

In [234]:
print("=" * 60)
print("ACTIVITY ANALYSIS")
print("=" * 60)

print(f"Busiest Day        : {busiest_day.strftime('%d %b %Y')}")
print(f"Messages on Day    : {day_messages}")

hour_label = f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00"     # One-hour interval is prepared

print(f"Busiest Hour       : {hour_label}")
print(f"Messages in Hour   : {hour_messages}")

ACTIVITY ANALYSIS
Busiest Day        : 04 May 2024
Messages on Day    : 74
Busiest Hour       : 18:00 - 19:00
Messages in Hour   : 244


**Verifying Activity Analysis**

In [235]:
print("=" * 60)
print("FEATURE 3 VERIFICATION")
print("=" * 60)

print(f"Days Analysed      : {len(daily_activity)}")
print(f"Hours Analysed     : {len(hourly_activity)}")
print(f"Peak Day Messages  : {day_messages}")
print(f"Peak Hour Messages : {hour_messages}")

FEATURE 3 VERIFICATION
Days Analysed      : 60
Hours Analysed     : 24
Peak Day Messages  : 74
Peak Hour Messages : 244


✔ **Feature 3 Completed Successfully**

### **Feature 4: Activity Heatmap**

This feature creates a participant-wise activity heatmap using a NumPy
matrix. The matrix stores the number of messages sent by each participant
during every hour of the day. The activity levels are represented using
text-based block characters as specified in the project constraints.

**Creating the Heatmap Matrix**

In [236]:
# Arranging participant names alphabetically
participants_list = sorted(participants)

# Creating participant index mapping
participant_index = {}

for index, name in enumerate(participants_list):
    participant_index[name] = index                  # Matrix row index is assigned

# Creating NumPy activity matrix
heatmap = np.zeros((len(participants_list), 24), dtype=int)

# Filling the activity matrix
for record in chat_data:

    row = participant_index[record["sender"]]
    column = record["timestamp"].hour

    heatmap[row][column] += 1                        # Hourly activity is recorded

**Converting Activity into Heat Levels**

In [237]:
# Finding the highest activity value
maximum_activity = np.max(heatmap)

# Converting message count into heat symbols
def get_heat_symbol(count):

    if count == 0:
        return "."

    elif count <= maximum_activity * 0.25:
        return "░"

    elif count <= maximum_activity * 0.60:
        return "▒"

    else:
        return "█"

**Displaying the Activity Heatmap**

In [238]:
print("=" * 95)
print("NUMPY ACTIVITY HEATMAP")
print("=" * 95)

print(f"{'Participant':<15}", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")                    # Hour labels are formatted

print()

print("-" * 95)

for row, participant in enumerate(participants_list):

    print(f"{participant:<15}", end="")

    for hour in range(24):

        symbol = get_heat_symbol(heatmap[row][hour])

        print(f" {symbol} ", end="")                # Activity level is displayed

    print()

NUMPY ACTIVITY HEATMAP
Participant    00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
-----------------------------------------------------------------------------------------------
Aman            ▒  █  █  ▒  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▒ 
Karan           .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ▒  ░  ▒  ▒  ▒  ▒  ░  ▒  ░  ░  ░  ░ 
Neha            .  .  .  .  .  ░  ░  ░  ▒  ▒  ▒  ░  ▒  ▒  ▒  ░  ▒  ▒  █  ▒  ▒  ▒  ▒  ▒ 
Priya           .  .  .  .  .  .  ░  ░  ▒  █  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ░  ░ 
Rahul           ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▒  ▒  ▒  ▒  █  ▒  █  █  ▒  █  ▒  ▒ 
Vikas           .  .  .  .  .  .  .  ░  ░  ░  ░  .  ░  ░  .  ░  ░  ░  ░  ░  ░  ░  ░  ░ 


In [239]:
print("\nLegend")
print(".  : No activity")
print("░  : Low activity")
print("▒  : Medium activity")
print("█  : High activity")


Legend
.  : No activity
░  : Low activity
▒  : Medium activity
█  : High activity


**Verifying Heatmap**

In [240]:
print("=" * 60)
print("FEATURE 4 VERIFICATION")
print("=" * 60)

print(f"Matrix Shape        : {heatmap.shape}")
print(f"Total Messages      : {np.sum(heatmap)}")
print(f"Maximum Cell Value  : {maximum_activity}")
print(f"Participants        : {len(participants_list)}")

FEATURE 4 VERIFICATION
Matrix Shape        : (6, 24)
Total Messages      : 3127
Maximum Cell Value  : 102
Participants        : 6


✔ **Feature 4 Completed Successfully**

### **Feature 5: Word Frequency Analysis**

This feature analyzes the most frequently used words in the WhatsApp group.
Common stop-words, media messages, and deleted messages are excluded to
identify meaningful communication patterns.

**Defining Stop Words**

In [241]:
# Defining common stop words
stop_words = {
    "the","a","an","is","am","are","was","were",
    "to","of","in","on","at","for","and","or",
    "i","you","me","my","we","our","your",
    "it","its","this","that","these","those",
    "be","been","have","has","had","do","does",
    "did","with","as","by","from","but","if",
    "not","so","can","will","just","ok","okay",
    "ya","haan","ha","lol","bro","hai",
    "how","what","when","where","why","who",
    "he","she","his","her","him","their",
    "they","them","which","about",
    "yes","no","yeah","please","thanks",
    "thank","very","all","any","everyone"
}

**Counting Word Frequencies**

In [242]:
# Initializing word frequency dictionary
word_frequency = {}

# Counting meaningful words
for record in chat_data:
    message = record["message"].lower()
    words = message.split()
    for word in words:
        word = word.strip(".,!?()[]{}:;\"'")      # Basic punctuation is removed
        if len(word) < 2:
            continue                              # Very short words are ignored
        if word in stop_words:
            continue                              # Stop words are excluded
        if word not in word_frequency:
            word_frequency[word] = 0

        word_frequency[word] += 1                 # Word frequency is updated

**Identifying Top Words**

In [243]:
# Sorting words by frequency
top_words = sorted(
    word_frequency.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]

** Displaying Top Words**

In [244]:
print("=" * 70)
print("TOP 10 MOST FREQUENT WORDS")
print("=" * 70)

print(f"{'Word':<15}{'Count':>8}   {'Frequency'}")
print("-" * 70)

maximum_count = top_words[0][1]                  # Highest frequency is obtained

for word, count in top_words:

    bar_length = int((count / maximum_count) * 30)
    bar = "█" * bar_length

    print(f"{word:<15}{count:>8}   {bar}")

TOP 10 MOST FREQUENT WORDS
Word              Count   Frequency
----------------------------------------------------------------------
guys                318   ██████████████████████████████
today               257   ████████████████████████
telling             179   ████████████████
up                  172   ████████████████
bhai                160   ███████████████
one                 157   ██████████████
started             150   ██████████████
scene               145   █████████████
entire              145   █████████████
anyone              139   █████████████


**Verifying Word Frequency**

In [245]:
print("=" * 60)
print("FEATURE 5 VERIFICATION")
print("=" * 60)

print(f"Unique Words      : {len(word_frequency)}")
print(f"Most Used Word    : {top_words[0][0]}")
print(f"Occurrences       : {top_words[0][1]}")

FEATURE 5 VERIFICATION
Unique Words      : 520
Most Used Word    : guys
Occurrences       : 318


**Identifying Top Words of Every Participant**

In [246]:
# Initializing participant word frequencies
participant_words = {}

# Counting words for every participant
for sender in participants:

    participant_words[sender] = {}

for record in chat_data:

    sender = record["sender"]

    words = record["message"].lower().split()

    for word in words:

        word = word.strip(".,!?()[]{}:;\"'")      # Basic punctuation is removed

        if len(word) < 2:
            continue                              # Very short words are ignored

        if word in stop_words:
            continue                              # Stop words are excluded

        if word not in participant_words[sender]:
            participant_words[sender][word] = 0

        participant_words[sender][word] += 1      # Participant frequency is updated

**Displaying Participant-wise Top Words**

In [247]:
print("=" * 70)
print("TOP 5 WORDS OF EACH PARTICIPANT")
print("=" * 70)

for sender in sorted(participants):

    print(f"\n{sender}")

    print("-" * 40)

    top_five = sorted(
        participant_words[sender].items(),
        key=lambda item: item[1],
        reverse=True
    )[:5]

    for word, count in top_five:

        print(f"{word:<15}{count:>5}")

TOP 5 WORDS OF EACH PARTICIPANT

Aman
----------------------------------------
sleep             71
i'm               56
anyone            49
wonder            43
night             41

Karan
----------------------------------------
telling          179
today            171
up               164
started          150
entire           145

Neha
----------------------------------------
guys             101
cant              58
today             48
way               48
wait              48

Priya
----------------------------------------
aman              93
anyone            90
take              72
guys              65
eat               60

Rahul
----------------------------------------
bhai             159
scene            144
kya              133
yaar             105
ja               101

Vikas
----------------------------------------
haha               4
sorry              3
busy               3
tha                3
exam               2


✔ **Feature 5 Completed Successfully**

### **Feature 6: Response Time Analysis**

This feature analyzes the average response time between consecutive user
messages. System messages are ignored, and only replies from different
participants are considered.

**Calculating Response Time**

In [248]:
# Initializing response time records
response_times = {}
previous_record = None                              # Previous message is initialized

# Traversing through chat records
for record in chat_data:

    if previous_record is None:
        previous_record = record
        continue
    if record["sender"] != previous_record["sender"]:
        time_difference = (
            record["timestamp"] - previous_record["timestamp"]
        ).total_seconds() / 60                      # Response time is calculated
        if time_difference >= 0:
            sender = record["sender"]
            if sender not in response_times:
                response_times[sender] = []
            response_times[sender].append(time_difference)   # Response time is stored

    previous_record = record

**Calculating Average Response Time**

In [249]:
# Calculating average response time
average_response = {}

for sender in response_times:
    times = response_times[sender]
    average_response[sender] = (
        sum(times) / len(times)
    )                                               # Average response time is calculated

**Displaying Response Pattern Analysis**

In [250]:
print("=" * 70)
print("RESPONSE PATTERN ANALYSIS")
print("=" * 70)

print(f"{'Member':<15}{'Avg Response (Minutes)':>25}")
print("-" * 70)

for sender, average in sorted(
        average_response.items(),
        key=lambda item: item[1]):

    print(f"{sender:<15}{average:>22.2f}")

RESPONSE PATTERN ANALYSIS
Member            Avg Response (Minutes)
----------------------------------------------------------------------
Vikas                           34.90
Rahul                           35.17
Karan                           36.83
Neha                            41.30
Priya                           42.57
Aman                            54.91


**Calculating Longest Silent Streak**

In [251]:
# Initializing silent streak records
silent_streak = {}

# Calculating longest inactivity period
for sender in participants:

    message_dates = []

    for record in chat_data:

        if record["sender"] == sender:

            message_dates.append(record["timestamp"].date())

    longest_gap = 0

    for index in range(1, len(message_dates)):

        gap = (message_dates[index] - message_dates[index - 1]).days

        if gap > longest_gap:
            longest_gap = gap                      # Longest gap is updated

    silent_streak[sender] = longest_gap

**Displaying Silent Streak Analysis**

In [252]:
print("=" * 70)
print("LONGEST SILENT STREAKS")
print("=" * 70)

print(f"{'Member':<15}{'Days':>10}")
print("-" * 70)

for sender, days in sorted(
        silent_streak.items(),
        key=lambda item: item[1],
        reverse=True):

    print(f"{sender:<15}{days:>10}")

LONGEST SILENT STREAKS
Member               Days
----------------------------------------------------------------------
Vikas                  12
Neha                    1
Karan                   1
Priya                   1
Aman                    1
Rahul                   1


**Verifying Response Time Analysis**

In [253]:
print("=" * 60)
print("FEATURE 6 VERIFICATION")
print("=" * 60)
fastest = min(average_response, key=average_response.get)
slowest = max(average_response, key=average_response.get)

print(f"Fastest Responder : {fastest}")
print(f"Average Time      : {average_response[fastest]:.2f} minutes")
print()

print(f"Slowest Responder : {slowest}")
print(f"Average Time      : {average_response[slowest]:.2f} minutes")
print()

longest_gap = max(silent_streak, key=silent_streak.get)
print(f"Longest Silent Streak : {longest_gap}")
print(f"Days Inactive         : {silent_streak[longest_gap]} days")

FEATURE 6 VERIFICATION
Fastest Responder : Vikas
Average Time      : 34.90 minutes

Slowest Responder : Aman
Average Time      : 54.91 minutes

Longest Silent Streak : Vikas
Days Inactive         : 12 days


✔ **Feature 6 Completed Successfully**

### **STEP 7 — Personality Archetypes Analysis**

This feature helps us to identify the communication style of every participant by analysing message activity, response behaviour, and average message length. Different archetypes are assigned based on the observed chat patterns.

**Calculating Participant Metrics**

In [254]:
# Initializing participant metrics
average_words = {}

# Calculating average words per message
for sender in participants:
    total_words = 0
    total_messages = 0

    for record in chat_data:
        if record["sender"] != sender:
            continue                              # Other participants are ignored
        total_messages += 1
        total_words += len(record["message"].split())

    if total_messages > 0:
        average_words[sender] = total_words / total_messages
    else:
        average_words[sender] = 0

**Assigning Personality Archetypes**

In [255]:
# Initializing participant archetypes
archetypes = {}

# Identifying key participants
most_active = max(message_count, key=message_count.get)
fastest_responder = min(average_response, key=average_response.get)
story_teller = max(average_words, key=average_words.get)
silent_observer = max(silent_streak, key=silent_streak.get)

# Assigning archetypes
for sender in participants:
    if sender == most_active:
        archetypes[sender] = "Conversation Starter"
    elif sender == story_teller:
        archetypes[sender] = "Story Teller"
    elif sender == silent_observer:
        archetypes[sender] = "Silent Observer"
    elif sender == fastest_responder:
        archetypes[sender] = "Fast Responder"
    else:
        archetypes[sender] = "Active Participant"     # Default archetype is assigned

**Displaying Personality Archetypes**

In [256]:
print("=" * 70)
print("PERSONALITY ARCHETYPES")
print("=" * 70)

print(f"{'Member':<15}{'Archetype'}")
print("-" * 70)

for sender in sorted(participants):

    print(f"{sender:<15}{archetypes[sender]}")

PERSONALITY ARCHETYPES
Member         Archetype
----------------------------------------------------------------------
Aman           Active Participant
Karan          Story Teller
Neha           Active Participant
Priya          Active Participant
Rahul          Conversation Starter
Vikas          Silent Observer


**Verifying Personality Archetypes**

In [257]:
print("=" * 60)
print("FEATURE 7 VERIFICATION")
print("=" * 60)
print(f"Conversation Starter : {most_active}")
print(f"Fastest Responder    : {fastest_responder}")
print(f"Story Teller         : {story_teller}")
print(f"Silent Observer      : {silent_observer}")

FEATURE 7 VERIFICATION
Conversation Starter : Rahul
Fastest Responder    : Vikas
Story Teller         : Karan
Silent Observer      : Vikas


✔ **Feature 7 Completed Successfully**

### **Feature 8: Final Report**

This feature is used to present the complete GroupDNA analysis in a single formatted report using the outputs generated from all previous features.

**Displaying Final Report**

In [258]:
print("=" * 70)
print("GROUPDNA FINAL REPORT")
print("=" * 70)

# Overall summary is displayed

print(f"Chat Duration      : {start_date.strftime('%d %b %Y')} to {end_date.strftime('%d %b %Y')}")
print(f"Total Days         : {(end_date.date() - start_date.date()).days + 1}")
print(f"Total Messages     : {total_messages}")
print(f"Participants       : {total_participants}")

print()

# Activity summary is displayed

print(f"Busiest Day        : {busiest_day.strftime('%d %b %Y')}")
print(f"Messages on Day    : {day_messages}")
print(f"Busiest Hour       : {busiest_hour:02d}:00 - {busiest_hour + 1:02d}:00")
print(f"Messages in Hour   : {hour_messages}")

print()
print("=" * 70)
print("PARTICIPANT CONTRIBUTIONS")
print("=" * 70)

highest_messages = max(message_count.values())

for sender, count in sorted(message_count.items(), key=lambda x: x[1], reverse=True):

    percentage = (count / total_messages) * 100
    bar = "█" * int((count / highest_messages) * 30)

    print(f"{sender:<15}{count:>5} ({percentage:5.2f}%)  {bar}")

print()
print("=" * 70)
print("TOP 10 WORDS")
print("=" * 70)

print(f"{'Word':<15}{'Count':>8}")
print("-" * 70)

# Frequent words are displayed

for word, count in top_words:

    print(f"{word:<15}{count:>8}")

print()
print("=" * 70)
print("RESPONSE SUMMARY")
print("=" * 70)

fastest = min(average_response, key=average_response.get)
slowest = max(average_response, key=average_response.get)
silent = max(silent_streak, key=silent_streak.get)

print(f"Fastest Responder  : {fastest} ({average_response[fastest]:.2f} minutes)")
print(f"Slowest Responder  : {slowest} ({average_response[slowest]:.2f} minutes)")
print(f"Silent Observer    : {silent} ({silent_streak[silent]} days)")

print()
print("=" * 70)
print("PERSONALITY ARCHETYPES")
print("=" * 70)

# Personality summary is displayed

for sender in sorted(participants):

    print(f"{sender:<15}{archetypes[sender]}")

print()
print("=" * 70)
print("End of Report")
print("=" * 70)

GROUPDNA FINAL REPORT
Chat Duration      : 01 Apr 2024 to 30 May 2024
Total Days         : 60
Total Messages     : 940
Participants       : 6

Busiest Day        : 04 May 2024
Messages on Day    : 74
Busiest Hour       : 18:00 - 19:00
Messages in Hour   : 244

PARTICIPANT CONTRIBUTIONS
Rahul            940 (100.00%)  ██████████████████████████████
Priya            712 (75.74%)  ██████████████████████
Neha             624 (66.38%)  ███████████████████
Aman             484 (51.49%)  ███████████████
Karan            345 (36.70%)  ███████████
Vikas             22 ( 2.34%)  

TOP 10 WORDS
Word              Count
----------------------------------------------------------------------
guys                318
today               257
telling             179
up                  172
bhai                160
one                 157
started             150
scene               145
entire              145
anyone              139

RESPONSE SUMMARY
Fastest Responder  : Vikas (34.90 minutes)
Slowest Respo

**Verifying Final Report**

In [259]:
print("=" * 60)
print("FEATURE 8 VERIFICATION")
print("=" * 60)
print("Group Summary           : Displayed")
print("Activity Summary        : Displayed")
print("Participant Statistics  : Displayed")
print("Word Analysis           : Displayed")
print("Response Summary        : Displayed")
print("Personality Summary     : Displayed")
print("Final Report            : Generated Successfully")

FEATURE 8 VERIFICATION
Group Summary           : Displayed
Activity Summary        : Displayed
Participant Statistics  : Displayed
Word Analysis           : Displayed
Response Summary        : Displayed
Personality Summary     : Displayed
Final Report            : Generated Successfully


✔ **Feature 8 Completed Successfully**

# ----------------------------- End Of Project -----------------------------